# Lab 09 — Filter an EEG — and prove delta survived

> **Type-2 lab — student version.** Fill in each `# TODO` (the answer cells raise `NotImplementedError` until you do), answer the **Checkpoint** questions, and complete the reflection cell, then re-run top to bottom. The instructor solution lives in `labs/solutions/` and is not included here.

**Covers.** Chapter 9 — §9.4–§9.6 (FIR vs IIR, zero-phase filtering and edge transients, the EEG band-pass + 50 Hz notch).

**Biomedical question.** Can I remove drift, muscle, and mains from sleep EEG without damaging the delta rhythm that defines deep sleep?
**Task type (§1.8).** Denoising/enhancement with a preservation constraint
**Information that must be preserved.** delta-band (0.5–4 Hz) amplitude and spindle (~11–16 Hz) morphology and timing
**Main assumptions.** the artifacts occupy bands mostly outside the features of interest
**Primary diagnostic.** compare delta band-power before/after; inspect the two-pass filtfilt response at 0.5 Hz
**Transfer challenge.** redesign for an ECG where the ST segment (not delta) must survive

*Self-contained: runs fully offline in a browser on a seeded synthetic signal — no data
files, no network. `load_signal()` calls a `load_from_open_dataset()` hook you can wire to a
real recording (wfdb / MNE / PhysioNet); it is **not** wired here, so the notebook announces
the fallback and continues.*

### Companion lab · *Biomedical Signal Processing & Data Analytics* (CM2013)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/CM2013/blob/main/labs/lab09_eeg_bandpass_notch_preserve_delta/lab09_eeg_bandpass_notch_preserve_delta.ipynb) [![nbviewer](https://img.shields.io/badge/view-nbviewer-orange)](https://nbviewer.org/github/farhad-abtahi/CM2013/blob/main/labs/lab09_eeg_bandpass_notch_preserve_delta/lab09_eeg_bandpass_notch_preserve_delta.ipynb) [![JupyterLite](https://img.shields.io/badge/run-JupyterLite-blue)](https://farhad-abtahi.github.io/CM2013/lite/lab/index.html?path=lab09_eeg_bandpass_notch_preserve_delta.ipynb)

In [ ]:
# --- shared setup (reproducible; offline fallback) ---
import numpy as np, matplotlib.pyplot as plt
from scipy import signal as sig
rng = np.random.default_rng(2013)
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True})

def load_from_open_dataset(kind, fs, secs):
    """Real-data hook — fill in for your track (wfdb / MNE / PhysioNet). Not wired by
    default, so this lab runs fully offline on synthetic data."""
    raise NotImplementedError("real-data hook not configured")

def load_signal(kind, fs, secs):
    """Return (x, fs). Try the real-data hook; else ANNOUNCE and use a synthetic stand-in."""
    try:
        return load_from_open_dataset(kind, fs, secs)
    except NotImplementedError as e:
        print(f"[load_signal] {e}; using synthetic fallback.")
        t = np.arange(int(fs*secs))/fs
        return synth(kind, t, fs), fs

# Injected delta tone with KNOWN amplitude, so its attenuation can be MEASURED directly
# (not merely inferred from a raw==filtered band-power tie). DELTA_F sits well inside the
# 0.5-40 Hz passband, so a correct zero-phase filter should return it almost untouched.
DELTA_F, DELTA_AMP = 1.5, 0.8

def synth(kind, t, fs):
    # sleep-like EEG: delta + spindle burst + drift + mains + broadband noise
    x = DELTA_AMP*np.sin(2*np.pi*DELTA_F*t)                        # delta (KNOWN amplitude)
    burst = (t > 5) & (t < 6.2)
    x += 0.4*np.sin(2*np.pi*13*t)*burst                            # spindle
    x += 0.6*np.sin(2*np.pi*0.2*t)                                 # baseline drift
    x += 0.15*np.sin(2*np.pi*50*t) + 0.25*rng.standard_normal(t.size)
    return x


In [ ]:
x, fs = load_signal("eeg", fs=200, secs=30)
t = np.arange(x.size)/fs

## 1. See it work — design the band-pass + notch (§9.6)
`# TODO` design a 4th-order Butterworth 0.5–40 Hz + a 50 Hz notch, apply zero-phase.

In [ ]:
# TODO band-pass 0.5-40 Hz + 50 Hz notch, built as second-order sections (SOS) for
# numerical robustness (a 4th-order band-pass as a single b/a is ill-conditioned at fs=200).
raise NotImplementedError("TODO: implement this — see the comment above")

## 1b. The primary diagnostic — one-pass vs two-pass response at 0.5 Hz
`filtfilt`/`sosfiltfilt` runs the filter **forward and backward**, so the magnitude response is **squared**: a one-pass −3 dB corner becomes −6 dB. The header calls this the primary diagnostic, so plot it — one-pass and two-pass magnitude across 0.1–5 Hz — and read the gain right at the 0.5 Hz corner, which is what decides whether delta survives.

In [ ]:
w, H = sig.sosfreqz(sos_chain, worN=8192, fs=fs)      # combined band-pass + notch response
mag1 = np.abs(H)                                     # one-pass (single application)
mag2 = mag1**2                                       # two-pass (filtfilt applies it twice)
m = (w >= 0.1) & (w <= 5)
plt.figure(figsize=(7.2, 3.2))
plt.plot(w[m], 20*np.log10(mag1[m] + 1e-12), label="one-pass (sosfilt)")
plt.plot(w[m], 20*np.log10(mag2[m] + 1e-12), label="two-pass (sosfiltfilt)")
plt.axvline(0.5, color="k", ls="--", lw=1, label="0.5 Hz corner")
plt.axhline(-3, color="gray", ls=":", lw=1); plt.axhline(-6, color="gray", ls=":", lw=1)
plt.xlabel("Hz"); plt.ylabel("magnitude (dB)"); plt.ylim(-24, 3)
plt.title("Two-pass squares the response: the -3 dB corner becomes -6 dB")
plt.legend(loc="lower right"); plt.tight_layout(); plt.show()
g1 = np.interp(0.5, w, mag1); g2 = np.interp(0.5, w, mag2)
print(f"gain at 0.5 Hz:     one-pass = {20*np.log10(g1):5.1f} dB   two-pass = {20*np.log10(g2):5.1f} dB")
print(f"gain at {DELTA_F} Hz (delta): one-pass = {20*np.log10(np.interp(DELTA_F, w, mag1)):5.1f} dB"
      f"   two-pass = {20*np.log10(np.interp(DELTA_F, w, mag2)):5.1f} dB  (delta essentially passed)")

## 2. Prove delta survived — the diagnostic
Compute delta (0.5–4 Hz) band-power before and after. It must be preserved.

In [ ]:
def bandpower(x, fs, lo, hi):
    f, P = sig.welch(x, fs=fs, nperseg=min(2048, x.size))
    return P[(f>=lo)&(f<hi)].sum()

def tone_amplitude(x, fs, f0):
    """Recover the amplitude of a pure tone at f0 by projecting onto cos/sin. Over an
    integer number of cycles the other components are ~orthogonal, so this reads the
    injected delta directly -> a MEASURED attenuation, not one inferred from a tie."""
    n = np.arange(x.size)
    c = np.cos(2*np.pi*f0*n/fs); s = np.sin(2*np.pi*f0*n/fs)
    return 2.0/x.size * np.hypot(x @ c, x @ s)

d_before, d_after = bandpower(x, fs, 0.5, 4), bandpower(y, fs, 0.5, 4)
print(f"delta power   raw={d_before:.3f}  filtered={d_after:.3f}  ratio={d_after/d_before:.2f}")

a_inj, a_raw, a_filt = DELTA_AMP, tone_amplitude(x, fs, DELTA_F), tone_amplitude(y, fs, DELTA_F)
print(f"delta AMPLITUDE  injected={a_inj:.3f}  raw={a_raw:.3f}  filtered={a_filt:.3f}"
      f"  filtered/injected={a_filt/a_inj:.3f}")
# Checkpoint: filtered/injected ~1 is now a MEASURED fact about the injected tone, not just
# a raw-vs-filtered coincidence. If a high-pass eats delta, this ratio drops (see below).

## 3. Make it fail — and diagnose the filtfilt trap (§9.5)
Raise the high-pass corner to 2 Hz (or note that `filtfilt` doubles attenuation at the corner). Recompute delta.

In [ ]:
# TODO high-pass at 2 Hz with filtfilt; recompute delta ratio and explain the loss
raise NotImplementedError("TODO: implement this — see the comment above")
# Checkpoint: filtfilt runs twice -> the effective -3 dB corner becomes -6 dB, and a 2 Hz
# corner puts the 1.5 Hz delta in the stopband. Where did delta go? The MEASURED ratio shows it.

### Live sanity check
A preservation claim you never verify is just a hope. These asserts run on the numbers computed above and encode the lab's four claims: the designed chain **passes** the injected delta tone (amplitude and band-power), the **two-pass** response really is the one-pass response *squared* (−3 dB → −6 dB at the corner), the notch actually **removes** mains while the spindle burst **survives**, and the naive 2 Hz high-pass **destroys** the very rhythm the question depends on.

In [ ]:
# --- live sanity check: every number below was computed by the cells above ---
# (1) what MUST survive: the injected 1.5 Hz delta tone comes back essentially untouched.
assert 0.90 <= a_filt / a_inj <= 1.10, f"delta amplitude must survive, got {a_filt/a_inj:.3f}"
assert 0.85 <= d_after / d_before <= 1.15, f"delta band-power ratio off: {d_after/d_before:.3f}"

# (2) filtfilt squares the magnitude response -- this is a law, not an approximation.
#     (g1 and g2 are each INTERPOLATED onto 0.5 Hz from the response grid, and interpolating
#      squares is not identical to squaring an interpolation, so allow 0.01 dB of grid slop.)
assert abs(20*np.log10(g2) - 2*20*np.log10(g1)) < 0.01, "two-pass must be one-pass SQUARED"
assert -7.0 < 20*np.log10(g2) < -5.0, f"two-pass gain at the 0.5 Hz corner should be ~-6 dB"

# (3) the artifacts the filter was built for are actually gone, and the OTHER feature of
#     interest (the ~13 Hz spindle burst) is not collateral damage.
mains_raw, mains_filt = tone_amplitude(x, fs, 50.0), tone_amplitude(y, fs, 50.0)
assert mains_filt < 0.05 * mains_raw, f"50 Hz notch failed: {mains_filt:.4f} vs {mains_raw:.4f}"
sp = (t > 5.0) & (t < 6.2)                       # the spindle burst window
sp_raw, sp_filt = tone_amplitude(x[sp], fs, 13.0), tone_amplitude(y[sp], fs, 13.0)
assert sp_filt > 0.90 * sp_raw, f"the spindle must survive too: {sp_filt:.3f} vs {sp_raw:.3f}"

# (4) the failure mode is real: the 2 Hz high-pass puts 1.5 Hz delta in the stopband.
assert r < 0.25, f"the 2 Hz high-pass must destroy delta, got ratio {r:.3f}"
assert bandpower(y_bad, fs, 0.5, 4) / d_before < 0.10, "delta band-power must collapse under the bad HP"

print("sanity check PASSED:")
print(f"  delta amplitude  {a_filt/a_inj:6.3f} x injected (good chain)"
      f"   vs {r:6.3f} x injected (2 Hz high-pass)")
print(f"  0.5 Hz corner    one-pass {20*np.log10(g1):5.1f} dB -> two-pass {20*np.log10(g2):5.1f} dB (squared)")
print(f"  50 Hz mains      {mains_raw:.4f} -> {mains_filt:.4f}   |  13 Hz spindle {sp_raw:.3f} -> {sp_filt:.3f}")

## Reflection

This reflection is for your own practice — there is nothing to submit. What matters is the *reasoning*, not hitting a particular number.

1. **Stable vs changed.** Which conclusion held no matter how you built the chain (order, SOS vs `b,a`, notch `Q`), and which number moved the moment you touched the high-pass corner?
2. **The preservation number.** Report `filtered/injected` for the 1.5 Hz delta tone under *both* designs, and say which single design parameter explains the difference.
3. **Read the response, not the plot.** At what frequency does the *two-pass* chain reach −3 dB, and why is that not the corner you asked `butter` for?
4. **New montage / new signal.** What would you measure before trusting this chain on a different EEG montage — and how would you redesign it for an ECG where the **ST segment**, not delta, is what must survive?

**Rule out.** Choosing a high-pass corner *near or inside the band you must keep* — and then applying it with `filtfilt` — is ruled out, not a matter of taste. Two-pass filtering **squares** the magnitude response, so the nominal −3 dB corner is really −6 dB and the roll-off eats further into the passband than the design sheet suggests; a 2 Hz corner therefore put the 1.5 Hz delta in the stopband and collapsed it to ~8% of its injected amplitude (band-power ratio ~0.01). That breaks the **§1.8** requirement that delta amplitude survive, so the recording can no longer answer the question it was made for. The disciplined move is to place the corner well below the lowest frequency the claim depends on, build the chain as **second-order sections** (a 4th-order band-pass as a single `b,a` is ill-conditioned at `fs = 200`), and then *measure* preservation on a known injected tone rather than eyeballing the waveform.

> *Your answers here.*